# Inference entry point -- demo (Colab)

The same shared pipeline core the ingest uses (`ct_preprocessing.pipeline.process_scan`), run the way a clinical
product would: one scan, or a folder of several. No labels, no manifest, nothing written to disk.

`INPUT_PATH` can be:
- a single NIfTI file (`.nii` / `.nii.gz`),
- a single DICOM series folder,
- or a folder holding several scans (several patients/series) -- every one found is processed; one bad or QC-failing scan does not stop the rest.

Each scan gets its own row: `passed=False` (no volume) on a hard error or a failed QC check, `passed=True` with the preprocessed
volume otherwise -- `(N, 224, 224)` int16 raw Hounsfield Units, exactly what the training cache holds. Windowing and feeding a
model are the model code's job.

## 1. Upload and extract the project zip

In [ ]:
import shutil
from pathlib import Path

zip_candidates = sorted(Path("/content").glob("*.zip"))
if not zip_candidates:
    from google.colab import files
    print("No zip found in /content yet -- pick your project zip to upload:")
    uploaded = files.upload()
    zip_candidates = [Path("/content") / name for name in uploaded.keys()]

zip_path = zip_candidates[0]
print("Using zip:", zip_path)

In [ ]:
import zipfile

PROJECT_DIR = Path("/content/ChestCT-Multi-Abnormality-Classifier")
extract_dir = Path("/content/_extracted")

for d in (PROJECT_DIR, extract_dir):
    if d.exists():
        shutil.rmtree(d)
extract_dir.mkdir()

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)

entries = list(extract_dir.iterdir())
if len(entries) == 1 and entries[0].is_dir():
    shutil.move(str(entries[0]), str(PROJECT_DIR))
else:
    shutil.move(str(extract_dir), str(PROJECT_DIR))

print("Project extracted to:", PROJECT_DIR)

## 2. Install

In [ ]:
%cd /content/ChestCT-Multi-Abnormality-Classifier
!pip install -q ".[dev]"
!python -c "import torch" 2>/dev/null || pip install -q torch

In [ ]:
try:
    import ct_preprocessing  # noqa: F401
    print("ct_preprocessing imported OK -- install succeeded in this runtime")
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "ct_preprocessing is not installed in THIS runtime. This is not a bug in the code -- "
        "it happens whenever the Colab runtime restarts or disconnects, which wipes every installed "
        "package even though the notebook still shows old cell output. "
        "Fix: re-run cells 1-2 above (upload/extract the zip, pip install), then run this cell again."
    ) from exc

If a DICOM archive later fails with "pixel data cannot be decoded" (compressed files), set this to `True`, run it, and re-run the ingest step.

In [ ]:
INSTALL_DICOM_CODECS = False #@param {type:"boolean"}
if INSTALL_DICOM_CODECS:
    !pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

## 3. Check the compute available

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = "auto" if torch.cuda.is_available() else "cpu"
print("preprocessing device:", DEVICE)

## 4. Point at a scan (or a folder of scans)

For a Drive path that was *shared with you*: in Google Drive, right-click the folder -> **Organize -> Add shortcut** -> My Drive first (Colab's mount only shows My Drive).

In [ ]:
INPUT_PATH = "/content/drive/MyDrive/NHRD - CT Scan/4214-26/P00001/S0001" #@param {type:"string"}

if INPUT_PATH.startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if not Path(INPUT_PATH).exists():
    raise FileNotFoundError(f"{INPUT_PATH!r} was not found")
print("input:", INPUT_PATH)

## 5. Run inference

In [ ]:
import dataclasses
from ct_preprocessing.config import load_config
from ct_preprocessing.inference import check_matches_cache, run_inference

# The settings and QC thresholds come from configs/preprocessing.yaml -- the file the cache was built with.
# The device is the one thing a run may change (CPU and GPU give the same array).
data_cfg = load_config()
cfg = dataclasses.replace(data_cfg.preprocess, device=DEVICE)
check_matches_cache(cfg, data_cfg)  # refuses to run if the YAML no longer matches the recorded cache

results = run_inference(INPUT_PATH, cfg=cfg, qc_thresholds=data_cfg.qc)
print(f"{len(results)} scan(s) found")
for r in results:
    if r.passed:
        print(f"  OK       {r.volume_id}: volume {r.hu.shape} {r.hu.dtype}, spacing {r.spacing_zyx} mm")
    elif r.error:
        print(f"  ERROR    {r.volume_id}: {r.error}")
    else:
        print(f"  REJECTED {r.volume_id}: failed QC -- {'; '.join(r.qc.reasons)}")

## 6. Results table

In [ ]:
import pandas as pd
rows = []
for r in results:
    rows.append({
        "volume_id": r.volume_id,
        "scan_path": r.scan_path,
        "format": r.format,
        "passed": r.passed,
        "reasons": "; ".join(r.qc.reasons) if r.qc else (r.error or ""),
        "flags": "; ".join(r.qc.flags) if r.qc else "",
    })
pd.DataFrame(rows)

## 7. Look at a slice

The returned volume is raw HU; the lung window is applied here only to display it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

first = next((r for r in results if r.passed), None)
if first is None:
    print("no scan passed QC")
else:
    sl = first.hu[first.hu.shape[0] // 2]
    plt.imshow(np.clip(sl, -1000, 400), cmap="gray")
    plt.title(f"{first.volume_id} -- middle slice, lung window")
    plt.axis("off")
    plt.show()